# Checking July-December 2010

AI SCI 290A

Here I check the models I chose in the earlier temperature, wind, and precipitation comparisons. The table below lists exactly which ones I use. The extra experiments in 03 and 05, such as nearby points and the neural network, are not tested in this notebook.

## 1. Models I chose

| What I predict | Model and inputs |
| --- | --- |
| Tomorrow's temperature | Linear regression: today's temperature, its 3-day and 7-day averages, u, and v |
| Tomorrow's u | Linear regression: today's u and v |
| Tomorrow's v | Linear regression: today's u and v, their 3-day and 7-day averages, and pressure |
| Whether tomorrow's tp is above zero | Logistic regression: today's tp, temperature, pressure, u, and v; probability cutoff 0.5 |

For temperature and wind, I compare MAE and RMSE with persistence. For positive-or-zero precipitation, I compare accuracy, precision, and recall with persistence and always-zero. I also count the correct predictions, missed positive days, and false alarms.

I train each model once using answers dated before July 2010. Then I check July-December without changing the inputs or the 0.5 cutoff based on these scores.

For precipitation, I predict whether the saved tp value is above zero. This is not a confirmed whole-day rain/no-rain answer. The test also covers only six months.

I make each prediction after that day's weather is available. As days pass, I can use the new observed weather to predict the following day and update the averages. I do not retrain the models during this period.

## 2. Load the original data

In [1]:
from pathlib import Path
import glob
import xarray as xr
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data" / "data"

In [2]:
temp_files = sorted(glob.glob(f"{DATA_DIR}/**/2m_temperature*.nc", recursive=True))
ds_temp = xr.open_mfdataset(temp_files, combine="by_coords")
temp_series = (ds_temp["t2m"].sel(latitude=33.75, longitude=-117.75, method="nearest") - 273.15).to_series()
print("temp files:", len(temp_files))

temp files: 12


In [3]:
u_files = sorted(glob.glob(f"{DATA_DIR}/**/10m_u_component_of_wind*.nc", recursive=True))
ds_u = xr.open_mfdataset(u_files, combine="by_coords")
u_series = (ds_u["u10"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("u files:", len(u_files))

u files: 12


In [4]:
v_files = sorted(glob.glob(f"{DATA_DIR}/**/10m_v_component_of_wind*.nc", recursive=True))
ds_v = xr.open_mfdataset(v_files, combine="by_coords")
v_series = (ds_v["v10"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("v files:", len(v_files))

v files: 12


In [5]:
pressure_files = sorted(glob.glob(f"{DATA_DIR}/**/mean_sea_level_pressure*.nc", recursive=True))
ds_pressure = xr.open_mfdataset(pressure_files, combine="by_coords")
pressure_series = (ds_pressure["msl"].sel(latitude=33.75, longitude=-117.75, method="nearest") / 100).to_series()
print("pressure files:", len(pressure_files))

pressure files: 12


In [6]:
tp_files = sorted(glob.glob(f"{DATA_DIR}/**/total_precipitation*.nc", recursive=True))
ds_tp = xr.open_mfdataset(tp_files, combine="by_coords")
tp_series = (ds_tp["tp"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("tp files:", len(tp_files))

tp files: 12


## 3. Make the daily table

I match the dates from the different files and check that no days are missing. Then I add tomorrow's answers and the averages. Each average uses today and earlier days.

In [7]:
assert temp_series.index.equals(u_series.index)
assert temp_series.index.equals(v_series.index)
assert temp_series.index.equals(pressure_series.index)
assert temp_series.index.equals(tp_series.index)

weather = pd.DataFrame({
    "today_temp": temp_series, "today_u": u_series, "today_v": v_series,
    "pressure_hpa": pressure_series, "today_tp": tp_series
})
assert weather.index.is_unique and weather.index.is_monotonic_increasing
assert weather.index.equals(pd.date_range("2009-01-01", "2010-12-31"))
assert not weather.isna().any().any()
print("Dates match; 730 daily rows with no missing values.")

Dates match; 730 daily rows with no missing values.


In [8]:
weather["temp_3day"] = weather["today_temp"].rolling(3).mean()
weather["temp_7day"] = weather["today_temp"].rolling(7).mean()
weather["u_3day"] = weather["today_u"].rolling(3).mean()
weather["u_7day"] = weather["today_u"].rolling(7).mean()
weather["v_3day"] = weather["today_v"].rolling(3).mean()
weather["v_7day"] = weather["today_v"].rolling(7).mean()

weather["tomorrow_temp"] = weather["today_temp"].shift(-1)
weather["tomorrow_u"] = weather["today_u"].shift(-1)
weather["tomorrow_v"] = weather["today_v"].shift(-1)
weather["tomorrow_tp"] = weather["today_tp"].shift(-1)
weather["target_date"] = weather.index + pd.Timedelta(days=1)

temp_features = ["today_temp", "temp_3day", "temp_7day", "today_u", "today_v"]
u_features = ["today_u", "today_v"]
v_features = ["today_u", "today_v", "u_3day", "u_7day", "v_3day", "v_7day", "pressure_hpa"]
signal_features = ["today_tp", "today_temp", "pressure_hpa", "today_u", "today_v"]

## 4. Choose the training and test dates

For temperature and wind, I use rows that have a complete 7-day average, just as in the last comparisons. The precipitation model does not use those averages, so it can keep the first six days. Each model and its baseline are checked on the same test dates.

I now include the earlier validation months in training. The scaler also learns from this training data only.

In [9]:
regression_data = weather.dropna(subset=temp_features + v_features + ["tomorrow_temp", "tomorrow_u", "tomorrow_v"]).copy()
signal_data = weather.dropna(subset=signal_features + ["tomorrow_tp"]).copy()
signal_data["today_signal"] = (signal_data["today_tp"] > 0).astype(int)
signal_data["tomorrow_signal"] = (signal_data["tomorrow_tp"] > 0).astype(int)

train_reg = regression_data[regression_data["target_date"] < "2010-07-01"].copy()
test_reg = regression_data[
    (regression_data["target_date"] >= "2010-07-01") &
    (regression_data["target_date"] < "2011-01-01")
].copy()
train_signal = signal_data[signal_data["target_date"] < "2010-07-01"].copy()
test_signal = signal_data[
    (signal_data["target_date"] >= "2010-07-01") &
    (signal_data["target_date"] < "2011-01-01")
].copy()

assert train_reg["target_date"].max() < test_reg["target_date"].min()
assert train_signal["target_date"].max() < test_signal["target_date"].min()
assert test_reg["target_date"].equals(test_signal["target_date"])
assert len(test_reg) == 184
print("Regression training rows:", len(train_reg))
print("Classification training rows:", len(train_signal))
print("Test rows for each target:", len(test_reg))
print("Test target dates:", test_reg["target_date"].min(), "to", test_reg["target_date"].max())

Regression training rows: 539
Classification training rows: 545
Test rows for each target: 184
Test target dates: 2010-07-01 00:00:00 to 2010-12-31 00:00:00


## 5. Train the chosen models and check the results

I use the choices listed above. The training answers are dated before July 2010. I then predict July-December using the same inputs and settings.

In [10]:
final_temp_model = LinearRegression()
final_temp_model.fit(train_reg[temp_features], train_reg["tomorrow_temp"])
final_u_model = LinearRegression()
final_u_model.fit(train_reg[u_features], train_reg["tomorrow_u"])
final_v_model = LinearRegression()
final_v_model.fit(train_reg[v_features], train_reg["tomorrow_v"])

test_reg["temp_prediction"] = final_temp_model.predict(test_reg[temp_features])
test_reg["u_prediction"] = final_u_model.predict(test_reg[u_features])
test_reg["v_prediction"] = final_v_model.predict(test_reg[v_features])

### Temperature: compare with persistence

In [11]:
temp_baseline_error = test_reg["today_temp"] - test_reg["tomorrow_temp"]
temp_model_error = test_reg["temp_prediction"] - test_reg["tomorrow_temp"]
temp_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["C", "C"],
    "MAE": [temp_baseline_error.abs().mean(), temp_model_error.abs().mean()],
    "RMSE": [((temp_baseline_error ** 2).mean()) ** 0.5, ((temp_model_error ** 2).mean()) ** 0.5]
})
temp_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,C,0.9983,1.3152
1,Linear regression,C,0.8962,1.1816


### u component: compare with persistence

In [12]:
u_baseline_error = test_reg["today_u"] - test_reg["tomorrow_u"]
u_model_error = test_reg["u_prediction"] - test_reg["tomorrow_u"]
u_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["m/s", "m/s"],
    "MAE": [u_baseline_error.abs().mean(), u_model_error.abs().mean()],
    "RMSE": [((u_baseline_error ** 2).mean()) ** 0.5, ((u_model_error ** 2).mean()) ** 0.5]
})
u_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,m/s,0.5319,0.8279
1,Linear regression,m/s,0.4684,0.6783


### v component: compare with persistence

In [13]:
v_baseline_error = test_reg["today_v"] - test_reg["tomorrow_v"]
v_model_error = test_reg["v_prediction"] - test_reg["tomorrow_v"]
v_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["m/s", "m/s"],
    "MAE": [v_baseline_error.abs().mean(), v_model_error.abs().mean()],
    "RMSE": [((v_baseline_error ** 2).mean()) ** 0.5, ((v_model_error ** 2).mean()) ** 0.5]
})
v_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,m/s,0.4974,0.7035
1,Linear regression,m/s,0.4503,0.6359


### Will the precipitation value be positive?

I keep the same logistic regression settings and 0.5 cutoff. The scaler learns from the training rows only.

In [14]:
final_scaler = StandardScaler()
X_train_signal = final_scaler.fit_transform(train_signal[signal_features])
X_test_signal = final_scaler.transform(test_signal[signal_features])
final_signal_model = LogisticRegression(max_iter=1000)
final_signal_model.fit(X_train_signal, train_signal["tomorrow_signal"])

test_signal["always_zero"] = 0
test_signal["persistence"] = test_signal["today_signal"]
test_signal["probability"] = final_signal_model.predict_proba(X_test_signal)[:, 1]
test_signal["logistic"] = (test_signal["probability"] >= 0.5).astype(int)
y_test_signal = test_signal["tomorrow_signal"]
print("Actual label counts:")
print(y_test_signal.value_counts().sort_index())

Actual label counts:
tomorrow_signal
0    106
1     78
Name: count, dtype: int64


In [15]:
always_zero_test_scores = {
    "Model": "Always zero",
    "Accuracy": accuracy_score(y_test_signal, test_signal["always_zero"]),
    "Precision": precision_score(y_test_signal, test_signal["always_zero"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["always_zero"], zero_division=0),
}

In [16]:
persistence_test_scores = {
    "Model": "Persistence",
    "Accuracy": accuracy_score(y_test_signal, test_signal["persistence"]),
    "Precision": precision_score(y_test_signal, test_signal["persistence"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["persistence"], zero_division=0),
}

In [17]:
logistic_test_scores = {
    "Model": "Logistic",
    "Accuracy": accuracy_score(y_test_signal, test_signal["logistic"]),
    "Precision": precision_score(y_test_signal, test_signal["logistic"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["logistic"], zero_division=0),
}

In [18]:
signal_test_results = pd.DataFrame([always_zero_test_scores, persistence_test_scores, logistic_test_scores])
signal_test_results.round(4)

,Model,Accuracy,Precision,Recall
0,Always zero,0.5761,0.0000,0.0000
1,Persistence,0.6957,0.6410,0.6410
2,Logistic,0.7065,0.7857,0.4231


In [19]:
pd.DataFrame(
    confusion_matrix(y_test_signal, test_signal["logistic"], labels=[0, 1]),
    index=["Actual zero", "Actual positive"],
    columns=["Predicted zero", "Predicted positive"]
)

,Predicted zero,Predicted positive
Actual zero,97,9
Actual positive,45,33


## 6. What I found

Across the 184 days in July-December 2010, the temperature and wind models have lower MAE and RMSE than persistence.

| What I predict | Persistence MAE | Model MAE | Persistence RMSE | Model RMSE |
| --- | ---: | ---: | ---: | ---: |
| Temperature (C) | 0.9983 | 0.8962 | 1.3152 | 1.1816 |
| u (m/s) | 0.5319 | 0.4684 | 0.8279 | 0.6783 |
| v (m/s) | 0.4974 | 0.4503 | 0.7035 | 0.6359 |

The models help during these months. I would need more data to see whether they work as well in other years. The temperature model here includes wind, so it differs from the temperature-only model in notebook 02.

### Precipitation did not improve in every way

Logistic regression has slightly higher accuracy than persistence: 0.7065 compared with 0.6957. Its precision is also higher, 0.7857 compared with 0.6410. But recall falls to 0.4231 compared with 0.6410.

There are 78 positive days. Logistic regression finds 33, misses 45, and gives 9 false alarms. When it predicts positive, it is right more often, but it misses many positive days. The recall improvement in validation did not carry over here.

I keep the same 0.5 cutoff after seeing these results. These are predictions of positive saved tp values. The always-zero model never predicts positive, so its precision cannot be calculated; the table shows 0 for it.

### What I learned

Recent weather helped the temperature and wind models beat persistence here. Precipitation was harder to judge. Logistic regression gave fewer false alarms, but also missed more positive days.

These results are for the models listed at the start. I only have two years of data and still need to confirm the precipitation calculation.